<a href="https://colab.research.google.com/github/PM461/NLP/blob/main/esercizio5_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 🛠️ 1. Installazione delle librerie (solo su Colab)
!pip install transformers accelerate pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 56.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In questa sezione importiamo i moduli , verifichiamo il supporto della GPU , addestriamo il modello microsoft , lo tokenizziamo per una rappresentazione numerica e passiamo tutto ad una pipeline di addestramento.

In [2]:
# 2. Importazione dei moduli
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import pandas as pd
import os

#  3. Configurazione (evita problemi su alcune macchine)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

#  4. Verifica della GPU
print(" GPU disponibile:", torch.cuda.is_available())

#  5. Caricamento del modello Phi-3.5-mini-instruct
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=False
)

#  6. Caricamento del tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    trust_remote_code=True
)

#  7. Creazione della pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=20,
    temperature=0.2,
    top_p=1
)




 GPU disponibile: True


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Device set to use cuda:0


Carica manualmente il file di definizioni presente nella directory della consegna in modo da eseguire la restante parte dello script che attraverso il transformer impostato sy text generator cerca di dare un etichetta vedendo dutte le definizioni (come nell'esercizio 2-3)

In [3]:
# Caricamento CSV (modifica il percorso se serve)
df = pd.read_csv("file_definizioni.csv", sep=";", names=["ParolaChiave", "Definizione"])

etichette = []

for parola_chiave, gruppo in df.groupby("ParolaChiave"):
    # Concateno tutte le definizioni in un unico testo
    testo_completo = " ".join(gruppo["Definizione"].tolist())

    prompt = (
        f"Queste sono le definizioni della parola '{parola_chiave}':\n"
        f"{testo_completo}\n"
        "Suggerisci una singola parola che rappresenti il tema principale comune a tutte queste definizioni.\n"
        "Rispondi con una sola parola."
    )

    output = pipe(prompt)
    generated_text = output[0]["generated_text"]

    # Estraggo la risposta eliminando il prompt
    risposta = generated_text[len(prompt):].strip()
    parola_finale = risposta.split()[0] if risposta else ""

    etichette.append((parola_chiave, parola_finale))

# Stampa le etichette risultanti
print("\nEtichette generate per parola chiave:")
for k, v in etichette:
    print(f"{k}: {v}")

# Salvataggio su file
with open("etichette_per_parola.txt", "w", encoding="utf-8") as f:
    for k, v in etichette:
        f.write(f"{k}: {v}\n")


Etichette generate per parola chiave:
Ansia: Ansia
Bias cognitivo: Cognitivo
Pendrive: USB
Telecomando: Controllo
